# Auditoria inicial — dados efetivamente recebidos

Inspeção anterior às estimações. Nenhuma imputação, truncamento ou seleção por p-valores. Dados originais preservados.

In [1]:
from utilitarios import *
from IPython.display import display
d = ler()
print(f"{len(d)} linhas, {d.id.nunique()} municípios, {d.ano.min()}–{d.ano.max()}")

3384 linhas, 141 municípios, 2001–2024


## 1. Chaves, suporte e perdas

A grade tem 141 municípios e 24 anos; presença na grade não garante conversão observada. Bioma predominante não localiza a conversão.

In [2]:
raw = pd.read_csv(CSV, dtype={'codigo_ibge':str})
assert len(d)==3384 and d.id.nunique()==141
assert set(d.ano)==set(range(2001,2025))
assert not d.duplicated(['id','ano']).any()
suporte=d.groupby('bioma').agg(municipios=('id','nunique'),pares=('ano','size'),taxa_valida=('pp_agricultura','count'),fiscal_lag_valida=('fiscal','count'))
fixos=d.groupby('id').pp_agricultura.count().eq(24)
suporte['municipios_fixos']=d[d.id.isin(fixos[fixos].index)].groupby('bioma').id.nunique()
suporte.to_csv(PASTA/'auditoria_suporte.csv');display(suporte)
cobertura=d.groupby(['ano','bioma']).agg(n=('id','size'),taxa_valida=('pp_agricultura','count'),autos_validos=('autos_validos','count'),fiscal_lag=('fiscal','count'))
cobertura.to_csv(PASTA/'auditoria_cobertura.csv')
print('Contagens parciais de fiscalização:',d.parcial.value_counts(dropna=False).to_dict())

,municipios,pares,taxa_valida,fiscal_lag_valida,municipios_fixos
bioma,,,,,
Amazônia,76,1824,1306,873,39
Cerrado,61,1464,1412,519,55
Pantanal,4,96,54,52,1


Contagens parciais de fiscalização: {nan: 1556, False: 1526, True: 302}


## 2. Unidades, taxas e distribuições

Conversão em ha; taxas em fração. Preços reais em USD2024/t (grãos), USD2024/kg (carne), já defasados. Aptidão em kg de matéria seca/ha. Fiscalização em contagem, não valor pago. Não converter GAEZ em receita comercial. Ausência de flags de cobertura MapBiomas no CSV impede conferir diretamente a integridade dos mapas; a auditoria verifica consistência interna.

In [3]:
numeros=raw.select_dtypes('number')
resumo=numeros.describe(percentiles=[.01,.25,.5,.75,.99]).T
resumo['faltantes']=numeros.isna().sum();resumo['zeros']=numeros.eq(0).sum();resumo['negativos']=numeros.lt(0).sum();resumo['infinitos']=np.isinf(numeros).sum()
resumo.to_csv(PASTA/'auditoria_distribuicoes.csv')
raw.isna().sum().rename('faltantes').to_csv(PASTA/'auditoria_faltantes_todas_colunas.csv')
verificacoes=[]
for u in ['agricultura','pastagem']:
    q=d['taxa_'+u];h=d['ha_'+u]
    erro=(h/d.estoque-q).abs().max()
    assert erro < 1e-12 and not q.dropna().lt(0).any() and not q.dropna().gt(1).any()
    verificacoes.append({'destino':u,'n':q.count(),'zeros':q.eq(0).sum(),'erro_taxa_max':erro,'ha_negativos':h.lt(0).sum(),'denominador_nao_positivo':d.estoque.le(0).sum()})
display(pd.DataFrame(verificacoes));pd.DataFrame(verificacoes).to_csv(PASTA/'auditoria_taxas.csv',index=False)
print('Limite: filtros já aplicados na construção do CSV não podem ser reconstruídos só com este arquivo.')
pd.DataFrame([{'coluna_original':k,'alias':v} for k,v in MAPA.items()]).to_csv(PASTA/'dicionario_alias.csv',index=False)

,destino,n,zeros,erro_taxa_max,ha_negativos,denominador_nao_positivo
0,agricultura,2772,265,9.980081e-17,0,0
1,pastagem,2772,0,1.110223e-16,0,0


Limite: filtros já aplicados na construção do CSV não podem ser reconstruídos só com este arquivo.


## 3. Absorção pelos efeitos fixos

Retirada iterativa de médias municipal e anual na amostra comum; não usar double-demean de uma passada no painel desbalanceado.

In [4]:
cols=['z_p_soja','z_p_milho','z_p_boi','z_a_soja','z_a_milho','expo_soja','expo_milho','fiscal','preco_fiscal']
s=d.dropna(subset=['pp_agricultura',*cols]).copy()
x=s[cols].copy()
for passo in range(1000):
    antes=x.to_numpy().copy()
    x=x-x.groupby(s.id).transform('mean')
    x=x-x.groupby(s.ano).transform('mean')
    if np.max(np.abs(x.to_numpy()-antes))<1e-12:break
else:raise RuntimeError('Projeção FE não convergiu')
absor=pd.DataFrame({'variavel':cols,'desvio_residual_twfe':x.std().values})
absor['absorvida']=absor.desvio_residual_twfe.lt(1e-10)
absor.to_csv(PASTA/'auditoria_absorcao.csv',index=False);display(absor)
assert absor.iloc[:5].absorvida.all()
assert not absor.iloc[5:].absorvida.any()
assert np.linalg.matrix_rank(x[cols[5:]].values)==4
meta={'git_head':__import__('subprocess').check_output(['git','rev-parse','HEAD'],cwd=RAIZ,text=True).strip(),'csv_sha256':hashlib.sha256(CSV.read_bytes()).hexdigest(),'dimensoes':list(raw.shape),'semente':SEMENTE,'python':platform.python_version(),'hashes_textos':{f.name:hashlib.sha256(f.read_bytes()).hexdigest() for f in (RAIZ/'2_INTI/chapters/chap1/text').glob('*.tex')}}
(AUDIT/'metadados_estimacao.json').write_text(json.dumps(meta,indent=2))

,variavel,desvio_residual_twfe,absorvida
0,z_p_soja,1.047301e-14,True
1,z_p_milho,1.113743e-14,True
2,z_p_boi,2.732946e-14,True
3,z_a_soja,4.995621e-31,True
4,z_a_milho,2.847246e-30,True
5,expo_soja,8.170900e-01,False
6,expo_milho,7.644733e-01,False
7,fiscal,1.314194e+00,False
8,preco_fiscal,1.365387e+00,False


1006